In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam, SGD, lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from UNET_LIB.InceptionUnet import InceptionUNet
from Brats_Dataset import Brats_Dataset
from torch.utils.data import Dataset, Subset, DataLoader
from torch.utils.tensorboard import SummaryWriter
train_total=10240

group_spec={
    'perfe':160,
    'poly+':213,
    'poly-':572,
    'rough':868,
    'bbox_msk':1562,
    'sam_box':1562,
    'point_msk':1615,
    'MedSamBox':1562
}

mult_spec={
    'perfe':[1,2,4,8,16,32,64],
    'poly+':[1,2,4,8,16,32,train_total/group_spec['poly+']],
    'poly-':[1,2,4,8,16,train_total/group_spec['poly-']],
    'rough':[1,2,4,8,train_total/group_spec['rough']],
    'bbox_msk':[1,2,4,train_total/group_spec['bbox_msk']],
    'sam_box':[1,2,4,train_total/group_spec['sam_box']],
    'point_msk':[1,2,4,train_total/group_spec['point_msk']],
    'MedSamBox':[1,2,4,train_total/group_spec['sam_box']],
}

pretrained = 'pb'
assert pretrained in ['pb','fb','ub']


img_preprocess = transforms.Compose([    

    transforms.RandomEqualize(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0,1),

])


msk_preprocess = transforms.Compose([

])

test_loader = DataLoader(Brats_Dataset('./Brats20/','test','perfe', 'f_', \
                                        test_preprocess, msk_preprocess,flip=False, rot=False,crop=False),\
                         batch_size=64, shuffle=False, num_workers=32,pin_memory=True)
train_val0 = Brats_Dataset('./Brats20/','trainval','perfe','f_',\
                           img_preprocess, msk_preprocess,flip=False, rot=False, crop=False)

val_data = Subset(Brats_Dataset('./Brats20/','trainval','perfe','f_',\
                                img_preprocess, msk_preprocess,flip=False, rot=False, crop=False),\
                  list(range(train_total, len(train_val0))))

val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=32,pin_memory=True)

num_classes = 2
max_iter = 12001
interval = 300
loss_fn = nn.CrossEntropyLoss(ignore_index=255) 

In [2]:
for label_type in ['MedSamBox']: #['perfe','poly+','poly-','rough','bbox_msk']:
    group_size = group_spec[label_type]
    DATA = Brats_Dataset('./Brats20/','trainval',label_type,'f_', img_preprocess, msk_preprocess,crop=False)
    for multiplicity in reversed(mult_spec[label_type]):
        
        train_subset = Subset(DATA, list(range(round(multiplicity*group_size))))
        train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True, num_workers=32,pin_memory=True)
        
        iteration = 1
        min_loss = np.inf
        
        model = InceptionUNet(1, n_classes=num_classes)
        optimizer = Adam(model.parameters(),lr=1e-3,eps=0.1, weight_decay=1e-6)
        model = model.cuda()
        model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
       
        train_iter = iter(train_loader)
        model.train()
        train_loss = 0
        writer = SummaryWriter()
        writer.add_text('Variant', model_name)
        
        while iteration < max_iter:            
            
            optimizer.zero_grad()
            try:
                X,y = next(train_iter)
            except:
                train_iter = iter(train_loader)
                X,y = next(train_iter)
                
            yhat = model(X.contiguous().cuda())
            loss = loss_fn(yhat,y.cuda())
            loss.backward()
            optimizer.step()
            iteration += 1
            train_loss += loss.item()
            
            if iteration%interval == 0:
                train_loss /= interval
                writer.add_scalar('Loss/train', train_loss, iteration)
                train_loss = 0.
                
                model.eval()
                with torch.no_grad():
                    for eval_loader, set_name in zip([val_loader,test_loader],['Val','Test']):
                        
                        class_intersect = np.zeros((num_classes,),dtype='float')
                        class_union= np.zeros((num_classes,),dtype='float')
                        for idx,(X,y) in enumerate(eval_loader):
                            y = y.cuda().contiguous().flatten()
                            yhat = model(X.contiguous().cuda())
                            yhat_lab = torch.argmax(yhat, dim=1).flatten()

                            for j in range(num_classes):

                                y_bi = y == j
                                yhat_bi = yhat_lab == j
                                I = ((y_bi * yhat_bi).sum()).item()
                                U = (y_bi.sum() + yhat_bi.sum() - I).item()
                                assert I <= U
                                class_intersect[j] += I
                                class_union[j] += U

                        IOUs = class_intersect/class_union
                        for cls_id in range(len(IOUs)):
                            writer.add_scalar(f'{set_name}/cls{cls_id}', IOUs[cls_id],iteration)

                        if set_name == 'Val' and (-IOUs[1] < min_loss):
                            min_loss = -IOUs[1]
                            to_save ={'model_state_dict': model.state_dict()}

                            if pretrained=='fb':
                                to_save['scheduler_state_dict']= scheduler.state_dict()

                            torch.save(to_save, f'./model_checkpoints/{model_name}.pth')  
            model.train()

In [3]:
assert False

AssertionError: 

In [ ]:
./model_checkpoints/XUNet_BMedSamBox_m2_pb.pth

In [6]:
pretrained='pb'
performance={}
num_classes=2
test_preprocess = transforms.Compose([    

    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_loader = DataLoader( Brats_Dataset('./Brats20/','test','perfe',
                                              'f_', test_preprocess, msk_preprocess,flip=False, rot=False, crop=False), 
                         batch_size=64, shuffle=False, num_workers=8)

model = InceptionUNet(1, n_classes=num_classes)
model = model.cuda()

for label_type in ['MedSamBox']:
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
        print(model_name)
        checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
                y = y.flatten()

                yhat = model(X.contiguous().cuda())
                yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
                skip_id = np.argwhere(y == 255)
                yhat_lab[skip_id] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
                    
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Brats_Inter-Union_{pretrained}_MedSAMbox', performance)


XUNet_BMedSamBox_m1_pb
Working on label_type=MedSamBox:1


/tmp/ipykernel_1298009/1654187476.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')


XUNet_BMedSamBox_m2_pb
Working on label_type=MedSamBox:2
XUNet_BMedSamBox_m4_pb
Working on label_type=MedSamBox:4
XUNet_BMedSamBox_m7_pb
Working on label_type=MedSamBox:7


In [9]:
for key, value in performance['MedSamBox'].items():
    print(key, value[0][1]/value[1][1])

1 0.5863881844202284
2 0.6251611465371987
4 0.6224534533585763
6.555697823303457 0.6844320255365323


In [ ]:
####some old code --abandoned
assert False
for i in range(len(multiplicity_list)): 
    print(f'multiplicity:{multiplicity_list[i]}: ', np.mean(class_intersect[i]/class_union[i]),\
          np.std(class_intersect[i]/class_union[i]),
          np.mean(2*class_intersect[i]/(class_intersect[i]+class_union[i])),
          np.std(2*class_intersect[i]/(class_intersect[i]+class_union[i])))

In [ ]:
#snipping result

label_type='perfe'
test_loader = DataLoader(VOCSeg_Dataset('./VOC2012/','val','perfe',img_preprocess,msk_preprocess,crop=True,crop_size=500), batch_size=16, shuffle=False)

for multiplicity in mult_spec[label_type]:
        
    model = UNet(in_channels = 1, num_classes=num_classes)
    model = nn.DataParallel(model).cuda()
        
    model_name = f'DLab_V{label_type}_m{round(multiplicity)}_{pretrained}' 
    try:
        checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print('Loaded', model_name)
    except:
        print(f'Error loading {model_name}')
        assert False

    model.eval()
    with torch.no_grad():    
        class_intersect = np.zeros((num_classes,),dtype='float')
        class_union= np.zeros((num_classes,),dtype='float')

        for idx,(X,y) in enumerate(test_loader):
            y = y.flatten()

            yhat = model(X.contiguous().cuda())['out']
            yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
            skip_id = np.argwhere(y == 255)
            yhat_lab[skip_id] = 255

            for j in range(num_classes):

                y_bi = y == j
                yhat_bi = yhat_lab == j
                I = (y_bi * yhat_bi).sum()
                U = y_bi.sum() + yhat_bi.sum() - I
                assert I <= U
                class_intersect[j] += I
                class_union[j] += U

        IOUs = class_intersect/class_union
        val_loss=-np.mean(IOUs)
        print('IOU', -val_loss)
            